# How finely can FoldX order 432 combinatorial designs?

The 432 Track A designs were scored by FoldX and the top 15 reported. This notebook is the
whole run, end to end: structure identity, scoring, ranking, and then the question the
ranking raised - **how much of that ordering survives the scorer's own run-to-run spread?**

It is a measurement about the *candidate population*, not about FoldX. FoldX's documented
accuracy against experimental ddG is roughly 0.8-1.0 kcal/mol RMSE, so the run-to-run SD
measured below is smaller than its known error, and `BuildModel`'s stochasticity is why
`numberOfRuns` exists. Precision and accuracy are separate things: the AUROC panels
(direction 0.792, within-position 0.810, Chinery 0.716) say the mean tracks measured ddG,
and they stand. A scorer can be valid and imprecise at the same time.

**Every FoldX step is guarded.** They cost about 9 core-hours for the 432, plus 1,027 s for
the n=3 sample and 4,570 s for the n=20 top 50, and they are not re-run when their output is
already on disk. Set `FORCE_FOLDX = True` to override, which you should not do casually.

Everything here was originally run as separate scripts over ssh; they live in
`examples/foldx_precision/` and this notebook calls the same ones, then re-derives every
reported figure from the output files. The numbers it prints are the numbers in
`docs/article_numbers.md`.

In [1]:
import csv, hashlib, os, statistics, subprocess, sys
from collections import Counter
from itertools import combinations
from math import sqrt
from pathlib import Path

ROOT = Path("..").resolve()                 # the repository root; this notebook lives in notebooks/
SCRIPTS = ROOT / "examples/foldx_precision"

STRUCTURE   = ROOT / "data/structures/1N8Z_fv_trimmed_Repair.pdb"
CANDIDATES  = ROOT / "data/track_a_final/candidates.csv"          # 432 sets, one row per set
MUTATIONS   = ROOT / "runs/track_a_foldx/candidates_all432.csv"   # the same, one row per mutation
FOLDX       = ROOT / "runs/track_a_foldx/foldx.csv"               # 432 scored rows
RANKING     = ROOT / "runs/rerank/ranking_qualified_v1_432.csv"
RANKING_115 = ROOT / "runs/rerank/ranking_qualified_v1.csv"       # the shipped 115-row ranking
SD_N3       = ROOT / "runs/track_a_foldx/foldx_sd_sample.csv"     # 60 candidates, 3 runs each
SD_N20      = ROOT / "runs/track_a_foldx/foldx_runs20_top50.csv"  # top 50, 20 runs each
ALANINE     = ROOT / "trastuzumab_foldx_alanine_scan.csv"         # single mutants, for contrast

# The committed structure every FoldX number in this project was produced against.
STRUCTURE_SHA = "79ce088822d539a99517770ccb2d4b089ad25778a5529fb6d0e37fe92ddbaba9"

FORCE_FOLDX = False     # True re-runs the FoldX steps. Hours of compute. See the note above.
Z = 1.96                # two-sided 95%, uncorrected; a Bonferroni variant is shown at the end

def rows(path):
    with open(path, newline="") as handle:
        return list(csv.DictReader(handle))

def have(path, expect_rows=None):
    "Is this output already on disk, and the right size?"
    if not path.is_file():
        return False
    return expect_rows is None or len(rows(path)) == expect_rows

def run_script(name, env=None, label=""):
    "Run one of examples/foldx_precision/*.py from the repository root."
    cmd = [sys.executable, str(SCRIPTS / name)]
    print(f"running {name} {label}".strip())
    done = subprocess.run(cmd, cwd=ROOT, env={**os.environ, **(env or {})},
                          capture_output=True, text=True)
    print(done.stdout[-2000:] or done.stderr[-2000:])
    return done.returncode

print("repository root:", ROOT)

repository root: /home/kira/protecher


## Step 0 - which structure

FoldX ddG is meaningless across structures: a differently-repaired model gives different
numbers, which is why `design_pipeline/api.py` puts the content hash of
`DESIGN_PIPELINE_FOLDX_PDB` in the cache key. Before anything is scored or compared, the
structure on disk has to be the one the earlier scores were produced against.

In [2]:
digest = hashlib.sha256(STRUCTURE.read_bytes()).hexdigest()
print(f"{STRUCTURE.name}\n  on disk  {digest}\n  expected {STRUCTURE_SHA}")
assert digest == STRUCTURE_SHA, "structure differs from the one the scores were made against"
print("  -> identical, so every ddG below is on one structure")

1N8Z_fv_trimmed_Repair.pdb
  on disk  79ce088822d539a99517770ccb2d4b089ad25778a5529fb6d0e37fe92ddbaba9
  expected 79ce088822d539a99517770ccb2d4b089ad25778a5529fb6d0e37fe92ddbaba9
  -> identical, so every ddG below is on one structure


## Step 1 - the candidate table

`candidates.csv` is one row per design. The FoldX wrapper reads one row per *mutation*, so
the variant ids are expanded through the package's own `design_pipeline.variants.parse_id`
rather than a local string split - the parser that made every other variant table here.

In [3]:
if have(MUTATIONS) and not FORCE_FOLDX:
    print(f"{MUTATIONS.name} already present")
else:
    run_script("expand_432.py")

sets = rows(CANDIDATES)
per_mutation = rows(MUTATIONS)
print(f"{len(sets)} designs -> {len(per_mutation)} mutation rows")
print("substitutions per design:",
      dict(sorted(Counter(int(r["n_subs"]) for r in sets).items())))
positions = {(r["chain"], int(r["position"])) for r in per_mutation}
print(f"distinct positions moved: {len(positions)}",
      f"(chain A {sum(1 for c,_ in positions if c=='A')}, chain B {sum(1 for c,_ in positions if c=='B')})")

candidates_all432.csv already present
432 designs -> 2184 mutation rows
substitutions per design: {5: 413, 6: 14, 7: 5}
distinct positions moved: 23 (chain A 10, chain B 13)


## Step 2 - FoldX scoring (guarded)

`design_pipeline/adapters/_foldx_score.py` runs `BuildModel` `FOLDX_RUNS` times per design
and `AnalyseComplex` on each model and its wild-type copy, so ddG is an interaction-energy
difference. It is resumable: designs already in `foldx.csv` are skipped, which is how the
317 were added to the original 115 without re-paying for them.

This step needs a licensed FoldX binary and the environment from `env.sh`
(`FOLDX_BINARY`, `DESIGN_PIPELINE_FOLDX_PDB`). It is skipped whenever the 432 rows exist.

In [4]:
if have(FOLDX, 432) and not FORCE_FOLDX:
    print(f"{FOLDX.name}: 432 rows already scored - skipping ~9 core-hours of FoldX")
else:
    print("would run: _foldx_score.py --pdb $DESIGN_PIPELINE_FOLDX_PDB "
          f"--out_dir runs/track_a_foldx --mutations_csv {MUTATIONS.name}")
    print("set FORCE_FOLDX = True and source env.sh to actually run it")

scored = rows(FOLDX)
print(f"\n{len(scored)} scored, {sum(1 for r in scored if not r['ddg'].strip())} blank")
print("columns:", list(scored[0].keys()))
ddg = {r["variant_id"]: float(r["ddg"]) for r in scored}
print(f"ddg range {min(ddg.values()):+.3f} .. {max(ddg.values()):+.3f} kcal/mol")
print(f"ddg > 0 (weaker than wild type): {sum(1 for v in ddg.values() if v > 0)} of {len(ddg)}")

foldx.csv: 432 rows already scored - skipping ~9 core-hours of FoldX

432 scored, 0 blank
columns: ['variant_id', 'ddg', 'ddg_fold', 'n_runs']
ddg range -4.182 .. +7.932 kcal/mol
ddg > 0 (weaker than wild type): 261 of 432


## Step 3 - the ranking

`qualified_v1` is the profile the negative-control panel licenses, and it is one term:

$$\mathrm{score} = z(-\mathrm{foldx\_ddg})$$

`COULOMB_WEIGHT = 0.0` in `examples/track_a_chem_rank.py`, so no chemistry term enters.
Ordering is on raw ddG and the $z$ is presentation only - it is a monotone function of ddG,
so the ordering does not depend on the population the $z$ is taken over. The $z$ does: it
uses `statistics.pstdev`, matching the package's own `_z`, so the column is on the same
scale as every ranking file already on disk.

The repository's own ranker cannot do this: `track_a_chem_rank.py rank` requires `--rows`
from folds, and the 317 have no folds. That is an artifact of how the 115-row ranking was
built, not of the formula.

In [5]:
if have(RANKING, 432) and not FORCE_FOLDX:
    print(f"{RANKING.name} already present")
else:
    run_script("rank_432.py", label="(needs the pre-session foldx.csv backup as argv[1])")

ranked = rows(RANKING)
ddg_ranked = [float(r["foldx_ddg"]) for r in ranked]
print(f"{len(ranked)} ranked rows, profile {set(r['profile'] for r in ranked)}")
print(f"{'#':>3} {'name':<6}{'src':<6}{'ddg':>9}{'z':>8}")
for r in ranked[:15]:
    print(f"{r['rank']:>3} {r['name']:<6}{r['source']:<6}{float(r['foldx_ddg']):9.3f}{float(r['score']):8.3f}")

ranking_qualified_v1_432.csv already present
432 ranked rows, profile {'qualified_v1'}
  # name  src         ddg       z
  1 k003  kira     -4.182   2.098
  2 k001  kira     -4.142   2.082
  3 k049  kira     -4.008   2.028
  4 k006  kira     -3.972   2.014
  5 k010  kira     -3.655   1.887
  6 k081  kira     -3.626   1.875
  7 k052  kira     -3.201   1.705
  8 k050  kira     -3.196   1.703
  9 k038  kira     -3.195   1.703
 10 k032  kira     -3.083   1.658
 11 k028  kira     -3.065   1.651
 12 k016  kira     -2.945   1.603
 13 k015  kira     -2.884   1.578
 14 k063  kira     -2.831   1.557
 15 k034  kira     -2.830   1.556


## Step 4 - what the rerank answered

The 317 were scored because the original 115 had been chosen by sorting
`priority_or_additive`, a column holding priority for our designs and additive scores for
Kira's - two incommensurable scales sorted as one. That is a funnel bug, so the designs it
dropped belong in scope regardless of what scoring them finds.

In [6]:
meta = {r["variant_id"]: r for r in sets}
old115 = {r["variant_id"] for r in rows(RANKING_115)}
the317 = set(meta) - old115
ours = {v for v, m in meta.items() if m["source"] == "ours"}
order = {r["variant_id"]: int(r["rank"]) for r in ranked}

old_top15 = [r["variant_id"] for r in rows(RANKING_115)[:15]]
cut = ddg[old_top15[14]]
new_top15 = [r["variant_id"] for r in ranked[:15]]

print(f"the 115 {len(old115)}   the 317 {len(the317)}   'ours' {len(ours)}")
print(f"old rank-15 cutoff: {meta[old_top15[14]]['name']} at {cut:+.4f}")
print(f"of the 317, better than that cutoff: {sum(1 for v in the317 if ddg[v] < cut)}")
best317 = min(the317, key=lambda v: ddg[v])
print(f"best of the 317: {meta[best317]['name']} rank {order[best317]}, "
      f"{ddg[best317]:+.4f}, short by {ddg[best317]-cut:.4f}")
print(f"new top 15 vs old top 15: {len(set(new_top15) & set(old_top15))} of 15 shared, "
      f"identical order: {new_top15 == old_top15}")
print(f"of the 32 'ours' in the new top 15: {sum(1 for v in new_top15 if v in ours)}")
print(f"best 'ours': {meta[min(ours, key=lambda v: order[v])]['name']} at rank {min(order[v] for v in ours)}")

print("\npredicted improvement (ddg < 0) - PREDICTED, never measured:")
for label, group in (("ours", ours), ("kira selected", old115), ("kira unselected", set(meta) - ours - old115)):
    imp = [v for v in group if ddg[v] < 0]
    print(f"  {label:>16}: {len(imp)}/{len(group)} = {100*len(imp)/len(group):.0f}%")

from scipy.stats import spearmanr
pairs = [(float(meta[v]["priority_or_additive"]), ddg[v]) for v in old115]
rho, p = spearmanr([a for a, _ in pairs], [b for _, b in pairs])
print(f"\nSpearman(priority_or_additive, ddg) inside the selected 115: {rho:+.4f} (p={p:.2g})")
print(f"  median ddg: the 115 {statistics.median([ddg[v] for v in old115]):+.3f}"
      f"   the 317 {statistics.median([ddg[v] for v in the317]):+.3f}")

the 115 115   the 317 317   'ours' 32
old rank-15 cutoff: k034 at -2.8299
of the 317, better than that cutoff: 0
best of the 317: k128 rank 17, -2.7614, short by 0.0685
new top 15 vs old top 15: 15 of 15 shared, identical order: True
of the 32 'ours' in the new top 15: 0
best 'ours': o13 at rank 71

predicted improvement (ddg < 0) - PREDICTED, never measured:
              ours: 30/32 = 94%
     kira selected: 86/115 = 75%
   kira unselected: 55/285 = 19%



Spearman(priority_or_additive, ddg) inside the selected 115: -0.3219 (p=0.00045)
  median ddg: the 115 -1.344   the 317 +1.721


## Step 5 - precision at 3 runs (guarded)

The wrapper used to average its `FOLDX_RUNS` models and delete the per-run values, so
nothing on disk said how well-determined any ddG was. (It now writes a `ddg_sd` column;
this measurement is why.) `measure_sd.py` keeps the three shifts for a 60-candidate sample:
ranks 1-30, where the top-15 boundary lives, plus 30 spread over the rest so the median is
not a property of the best designs only.

In [7]:
if have(SD_N3, 60) and not FORCE_FOLDX:
    print(f"{SD_N3.name}: 60 rows already measured - skipping 1,027 s")
else:
    run_script("measure_sd.py", env={"FOLDX_RUNS": "3", "FOLDX_WORKERS": "8"})

sd3 = rows(SD_N3)
sds = sorted(float(r["ddg_sd"]) for r in sd3)
pct = lambda v, q: v[min(len(v)-1, int(q*len(v)))]
print(f"n={len(sds)}  median SD {statistics.median(sds):.4f}  p90 {pct(sds,0.90):.4f}  "
      f"range {sds[0]:.4f}..{sds[-1]:.4f}  SD==0: {sum(1 for s in sds if s==0)}")

drift = [abs(float(r["ddg_mean"]) - float(r["ddg_in_foldx_csv"])) for r in sd3]
print(f"remeasured mean vs the stored ddg: max |diff| {max(drift):.4f}  "
      "-> the pipeline is reproducible; the three runs simply disagree with each other")

ala = [float(r["ddg_bind_sd"]) for r in rows(ALANINE) if r["status"] == "ok" and r["ddg_bind_sd"].strip()]
print(f"\nsingle alanine mutants, same structure, same calls: n={len(ala)} "
      f"median SD {statistics.median(ala):.4f}  mean {statistics.mean(ala):.4f}")
print("-> a single-mutant precision estimate is off by more than an order of magnitude for")
print("   5-7 substitution designs: more substitutions give BuildModel more to rearrange.")

foldx_sd_sample.csv: 60 rows already measured - skipping 1,027 s
n=60  median SD 0.5643  p90 1.1340  range 0.0327..1.8000  SD==0: 0
remeasured mean vs the stored ddg: max |diff| 0.0000  -> the pipeline is reproducible; the three runs simply disagree with each other

single alanine mutants, same structure, same calls: n=210 median SD 0.0000  mean 0.0404
-> a single-mutant precision estimate is off by more than an order of magnitude for
   5-7 substitution designs: more substitutions give BuildModel more to rearrange.


## Step 6 - precision at 20 runs, top 50 (guarded)

At 3 runs the standard error of the mean is large relative to the gaps between adjacent
designs. 20 runs cuts it by $\sqrt{20/3} = 2.58$, which is what decides whether the top of
the ranking has resolvable structure in it.

In [8]:
if have(SD_N20, 50) and not FORCE_FOLDX:
    print(f"{SD_N20.name}: 50 rows already measured - skipping 4,570 s")
else:
    run_script("measure_runs20.py", env={"FOLDX_RUNS": "20", "FOLDX_WORKERS": "12", "TOP_N": "50"})

r20 = rows(SD_N20)
for r in r20:
    r["mean"], r["sd"], r["se"] = float(r["ddg_mean"]), float(r["ddg_sd"]), float(r["ddg_se"])
    r["mean_n3"] = float(r["ddg_mean_n3"])
r20.sort(key=lambda r: r["mean"])
print(f"n={len(r20)} at {r20[0]['n_runs']} runs")
print(f"  SD median {statistics.median([r['sd'] for r in r20]):.4f}"
      f"   SE median {statistics.median([r['se'] for r in r20]):.4f}")

foldx_runs20_top50.csv: 50 rows already measured - skipping 4,570 s
n=50 at 20 runs
  SD median 0.6470   SE median 0.1447


## Step 7 - how much did the ordering actually move?

**This is the number that decides how much the precision result is worth.** A high rank
correlation between the 3-run and 20-run orderings would mean the ranking was stable despite
the spread and the concern was largely theoretical. A low one means the ordering was being
driven by sampling.

Every resolution test below uses each candidate's **own** standard error, because the SD is
strongly candidate-specific - 0.03 to 1.80 at n=3. Two designs are resolvable when

$$|\bar{x}_i - \bar{x}_j| \ge 1.96\sqrt{\mathrm{se}_i^2 + \mathrm{se}_j^2}$$

In [9]:
from scipy.stats import kendalltau

rho, p_rho = spearmanr([int(r["rank_n3"]) for r in r20], range(1, len(r20)+1))
tau, _ = kendalltau([int(r["rank_n3"]) for r in r20], list(range(1, len(r20)+1)))
print(f"Spearman(3-run order, 20-run order) over the top {len(r20)}: {rho:+.4f}  (p={p_rho:.2g})")
print(f"Kendall tau: {tau:+.4f}")

by_n3 = sorted(r20, key=lambda r: int(r["rank_n3"]))
old15, new15 = {r["name"] for r in by_n3[:15]}, {r["name"] for r in r20[:15]}
print(f"\nranks changed: {sum(1 for i, r in enumerate(r20) if int(r['rank_n3']) != i+1)} of {len(r20)}")
print(f"top-15 kept: {len(old15 & new15)} of 15   entered {sorted(new15-old15)}   left {sorted(old15-new15)}")
cut_row = next(r for r in r20 if int(r["rank_n3"]) == 15)
print(f"the old rank-15 cutoff, {cut_row['name']}, is now rank {r20.index(cut_row)+1}")

Spearman(3-run order, 20-run order) over the top 50: +0.7567  (p=2.1e-10)
Kendall tau: +0.5739

ranks changed: 46 of 50
top-15 kept: 11 of 15   entered ['k027', 'k047', 'k069', 'k128']   left ['k028', 'k032', 'k034', 'k052']
the old rank-15 cutoff, k034, is now rank 21


In [10]:
def resolvable(a, b, z=Z):
    return abs(a["mean"] - b["mean"]) >= z * sqrt(a["se"]**2 + b["se"]**2)

band = [r for r in r20 if not resolvable(r, cut_row)]
pos = {r["name"]: i+1 for i, r in enumerate(r20)}
print(f"tied band around {cut_row['name']}: {len(band)} of {len(r20)} not resolvable from it")
print(f"  ranks {min(pos[r['name']] for r in band)}-{max(pos[r['name']] for r in band)}, "
      f"{max(r['mean'] for r in band) - min(r['mean'] for r in band):.3f} kcal/mol wide")

allpairs = list(combinations(r20, 2))
res = sum(1 for a, b in allpairs if resolvable(a, b))
print(f"pairwise: {res} of {len(allpairs)} resolvable ({100*res/len(allpairs):.1f}%)")

# Greedy tiers from the best design down. Pairwise "not resolvable" is not transitive, so
# connected components chain the whole list into one blob; the leader rule is the usable one.
tiers = []
for r in r20:
    if tiers and not resolvable(r, tiers[-1][0]):
        tiers[-1].append(r)
    else:
        tiers.append([r])
print(f"\nresolvable tiers: {len(tiers)}")
for i, t in enumerate(tiers, 1):
    print(f"  tier {i}: n={len(t):>2}  {t[0]['mean']:+.3f}..{t[-1]['mean']:+.3f}  "
          + ", ".join(x["name"] for x in t))

print(f"\nTOP TIER ({len(tiers[0])} designs)")
for r in tiers[0]:
    print(f"  {r['name']:<6}{r['mean']:8.3f}  SD {r['sd']:.3f}  SE {r['se']:.3f}  {r['variant_id']}")

from scipy.stats import norm
zb = float(norm.ppf(1 - 0.05/(2*len(allpairs))))
bonf = []
for r in r20:
    if bonf and not resolvable(r, bonf[-1][0], zb):
        bonf[-1].append(r)
    else:
        bonf.append([r])
print(f"\nwith Bonferroni over {len(allpairs)} pairs (z={zb:.3f}): {len(bonf)} tiers, top tier n={len(bonf[0])}")

tied band around k034: 21 of 50 not resolvable from it
  ranks 11-35, 0.904 kcal/mol wide
pairwise: 856 of 1225 resolvable (69.9%)

resolvable tiers: 8
  tier 1: n= 5  -3.850..-3.498  k003, k006, k049, k010, k001
  tier 2: n= 4  -3.209..-3.051  k038, k081, k050, k027
  tier 3: n= 3  -2.999..-2.809  k063, k016, k015
  tier 4: n=10  -2.775..-2.410  k069, k128, k047, k028, k161, k089, k052, k074, k034, k024
  tier 5: n= 8  -2.392..-2.038  k110, k108, k007, k045, k179, k008, k025, k032
  tier 6: n=12  -2.033..-1.678  k119, k151, k079, k048, k011, k136, k082, k088, k066, k058, k005, k041
  tier 7: n= 7  -1.638..-1.040  k021, k093, k044, k054, k033, k018, k039
  tier 8: n= 1  -0.991..-0.991  k037

TOP TIER (5 designs)
  k003    -3.850  SD 0.647  SE 0.145  A:D28W+A:S50R+A:Y92W+B:Y52R+B:Y105W
  k006    -3.839  SD 0.775  SE 0.173  A:D28W+A:S50R+A:Y92W+B:Y52R+B:D102W
  k049    -3.599  SD 0.681  SE 0.152  A:S50R+A:Y92W+B:Y52R+B:G101F+B:M107R
  k010    -3.571  SD 0.422  SE 0.094  A:D28W+A:S50R+B:D

### Selection bias in the reported values

The top 50 was chosen *by* its 3-run mean, so the selection favours designs whose three runs
happened to land low - and those are the high-variance ones. Re-measuring them is therefore
expected to move them in one direction.

In [11]:
from scipy.stats import pearsonr
shift = [r["mean"] - r["mean_n3"] for r in r20]
sd20 = [r["sd"] for r in r20]
print(f"shift (20-run mean minus 3-run mean): mean {statistics.mean(shift):+.4f} kcal/mol, "
      f"{sum(1 for s in shift if s > 0)} of {len(shift)} got worse")
print(f"corr(SD, shift): pearson {pearsonr(sd20, shift)[0]:+.4f}  spearman {spearmanr(sd20, shift)[0]:+.4f}")
lo = [s for s, d in zip(shift, sd20) if d < 0.4]
hi = [s for s, d in zip(shift, sd20) if d > 0.9]
print(f"  SD < 0.4 (n={len(lo)}): mean shift {statistics.mean(lo):+.4f}")
print(f"  SD > 0.9 (n={len(hi)}): mean shift {statistics.mean(hi):+.4f}")
print(f"best ddG: 3 runs said {min(r['mean_n3'] for r in r20):+.3f}, 20 runs says {min(r['mean'] for r in r20):+.3f}")

shift (20-run mean minus 3-run mean): mean +0.2565 kcal/mol, 38 of 50 got worse
corr(SD, shift): pearson +0.6990  spearman +0.6592
  SD < 0.4 (n=11): mean shift -0.0458
  SD > 0.9 (n=11): mean shift +0.7618
best ddG: 3 runs said -4.182, 20 runs says -3.850


## Step 8 - why the ordering was hard: the candidates, not the scorer

The 432 are combinatorial recombinations of a small set of positions, so they cluster. The
comparison that matters is the spread we are trying to resolve against the spread the scorer
was validated on - matched at the same number of variants, so density is not confounded with
how many were taken.

In [12]:
for k in (15, 22, 30, 50, len(ddg_ranked)):
    span = ddg_ranked[k-1] - ddg_ranked[0]
    print(f"DESIGNS top {k:>3}: span {span:6.3f} kcal/mol  ({span/(k-1):.4f} per adjacent pair)")

ev = rows(ROOT / "data/benchmarks/trastuzumab_mutation_evidence.csv")
adapt = {}
for r in ev:
    if r["study"].startswith("adapt") and r.get("ddg_reported_kcal_mol", "").strip():
        try: adapt[r["variant"]] = float(r["ddg_reported_kcal_mol"])
        except ValueError: pass
av = sorted(adapt.values())
print(f"\nBENCHMARK ADAPT: n={len(av)}  span {av[-1]-av[0]:.3f} kcal/mol "
      f"({(av[-1]-av[0])/(len(av)-1):.4f} per adjacent pair)")
d22 = ddg_ranked[21] - ddg_ranked[0]
print(f"at matched n=22: designs {d22:.3f} vs benchmark {av[-1]-av[0]:.3f} "
      f"-> {((av[-1]-av[0])/21)/(d22/21):.2f}x more tightly packed")

gaps = [ddg_ranked[i+1]-ddg_ranked[i] for i in range(29)]
print(f"\nmedian adjacent gap in the top 30: {statistics.median(gaps):.4f}")
print(f"a gap that size is resolvable only if SD < {statistics.median(gaps)/(Z*sqrt(2/3)):.4f}")
print(f"measured median SD at 3 runs: {statistics.median(sds):.4f}")

DESIGNS top  15: span  1.352 kcal/mol  (0.0966 per adjacent pair)
DESIGNS top  22: span  1.556 kcal/mol  (0.0741 per adjacent pair)
DESIGNS top  30: span  1.802 kcal/mol  (0.0622 per adjacent pair)
DESIGNS top  50: span  2.402 kcal/mol  (0.0490 per adjacent pair)
DESIGNS top 432: span 12.115 kcal/mol  (0.0281 per adjacent pair)

BENCHMARK ADAPT: n=22  span 2.790 kcal/mol (0.1329 per adjacent pair)
at matched n=22: designs 1.556 vs benchmark 2.790 -> 1.79x more tightly packed

median adjacent gap in the top 30: 0.0363
a gap that size is resolvable only if SD < 0.0227
measured median SD at 3 runs: 0.5643


## What this shows, and what it does not

- All 432 designs are FoldX-scored on one structure, 0 failures.
- **None of the 317 beat the old top-15 cutoff**, so the funnel bug cost nothing at the top -
  though the designs it dropped sit distinctly better than a typical Kira design.
- The 3-run ordering carries real signal but its fine structure does not survive
  re-measurement: the design that defined the rank-15 cutoff moves out of the top 15.
- At 20 runs the top 50 has genuine structure - most pairs are resolvable and there is a
  small top tier - but adjacent ranks are not separable at either sample size.
- Selecting on an imprecise estimate also **biases the values reported**, by about
  +0.26 kcal/mol here, concentrated in the noisiest candidates.

What it does not show: anything about these designs' actual affinity. Every number here is a
prediction. Nothing in this project measures a binding constant, and `ddg < 0` means
*predicted* to improve.

The transferable practice: **measure your scorer's precision on your own candidate
population, and compare it to the spread you are trying to resolve.** It is one short run,
and it is the difference between a ranking and a list.